In [2]:
!git clone https://github.com/jwkirchenbauer/lm-watermarking
%cd lm-watermarking
!pip install -r requirements.txt


Cloning into 'lm-watermarking'...
remote: Enumerating objects: 351, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 351 (delta 32), reused 16 (delta 16), pack-reused 294 (from 2)
Receiving objects: 100% (351/351), 12.01 MiB | 15.72 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/content/lm-watermarking/lm-watermarking


In [3]:
import torch
print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

Torch version: 2.10.0+cpu
GPU available: False


In [1]:
import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


In [2]:
import torch
print(torch.__version__)
print(torch.cuda.get_device_name(0))

2.10.0+cu128
Tesla T4


In [5]:
!ls

!git clone https://github.com/jwkirchenbauer/lm-watermarking
%cd lm-watermarking
!pip install -r requirements.txt


sample_data
Cloning into 'lm-watermarking'...
remote: Enumerating objects: 351, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 351 (delta 32), reused 16 (delta 16), pack-reused 294 (from 2)
Receiving objects: 100% (351/351), 12.01 MiB | 27.75 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/content/lm-watermarking


In [6]:
!ls

alternative_prf_schemes.py	 homoglyph_data  README.md
app.py				 homoglyphs.py	 requirements.txt
demo_watermark.py		 LICENSE.md	 setup.cfg
experiments			 MANIFEST.in	 watermark_processor.py
extended_watermark_processor.py  normalizers.py  watermark_reliability_release
hf_hub_space_demo		 pyproject.toml


In [8]:
!python demo_watermark.py

Namespace(run_gradio=True, demo_public=False, model_name_or_path='facebook/opt-6.7b', prompt_max_length=None, max_new_tokens=200, generation_seed=123, use_sampling=True, sampling_temp=0.7, n_beams=1, use_gpu=True, seeding_scheme='simple_1', gamma=0.25, delta=2.0, normalizers='', ignore_repeated_bigrams=False, detection_z_threshold=4.0, select_green_tokens=True, skip_model_load=False, seed_separately=True, load_fp16=False)
Namespace(run_gradio=True, demo_public=False, model_name_or_path='facebook/opt-6.7b', prompt_max_length=None, max_new_tokens=200, generation_seed=123, use_sampling=True, sampling_temp=0.7, n_beams=1, use_gpu=True, seeding_scheme='simple_1', gamma=0.25, delta=2.0, normalizers=[], ignore_repeated_bigrams=False, detection_z_threshold=4.0, select_green_tokens=True, skip_model_load=False, seed_separately=True, load_fp16=False)
Loading weights: 100% 516/516 [00:00<00:00, 912.91it/s, Materializing param=model.decoder.layers.31.self_attn_layer_norm.weight]
Traceback (most rec

In [9]:
!python demo_watermark.py --model_name_or_path gpt2

Namespace(run_gradio=True, demo_public=False, model_name_or_path='gpt2', prompt_max_length=None, max_new_tokens=200, generation_seed=123, use_sampling=True, sampling_temp=0.7, n_beams=1, use_gpu=True, seeding_scheme='simple_1', gamma=0.25, delta=2.0, normalizers='', ignore_repeated_bigrams=False, detection_z_threshold=4.0, select_green_tokens=True, skip_model_load=False, seed_separately=True, load_fp16=False)
Namespace(run_gradio=True, demo_public=False, model_name_or_path='gpt2', prompt_max_length=None, max_new_tokens=200, generation_seed=123, use_sampling=True, sampling_temp=0.7, n_beams=1, use_gpu=True, seeding_scheme='simple_1', gamma=0.25, delta=2.0, normalizers=[], ignore_repeated_bigrams=False, detection_z_threshold=4.0, select_green_tokens=True, skip_model_load=False, seed_separately=True, load_fp16=False)
config.json: 100% 665/665 [00:00<00:00, 2.72MB/s]
model.safetensors: 100% 548M/548M [00:05<00:00, 109MB/s]
Loading weights: 100% 148/148 [00:00<00:00, 2260.26it/s, Materializ

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# These class names exist in this repo
from watermark_processor import WatermarkLogitsProcessor, WatermarkDetector

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

prompt = "Explain in simple terms what an LLM watermark is and why it matters."

# Watermark params (same idea as demo)
gamma = 0.25
delta = 2.0
seeding_scheme = "simple_1"

# Build watermark objects
wm_processor = WatermarkLogitsProcessor(vocab=list(tokenizer.get_vocab().values()),
                                       gamma=gamma, delta=delta,
                                       seeding_scheme=seeding_scheme,
                                       select_green_tokens=True)

detector = WatermarkDetector(vocab=list(tokenizer.get_vocab().values()),
                             gamma=gamma,
                             seeding_scheme=seeding_scheme,
                             device=device,
                             tokenizer=tokenizer,
                             z_threshold=4.0,
                             select_green_tokens=True)

def generate_text(use_watermark: bool):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    logits_processors = [wm_processor] if use_watermark else None

    out = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.7,
        logits_processor=logits_processors
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

normal = generate_text(use_watermark=False)
watermarked = generate_text(use_watermark=True)

print("=== NORMAL ===")
print(normal[-800:])  # show last part
print("\nDetection (normal):", detector.detect(normal))

print("\n=== WATERMARKED ===")
print(watermarked[-800:])
print("\nDetection (watermarked):", detector.detect(watermarked))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== NORMAL ===
Explain in simple terms what an LLM watermark is and why it matters. It is a short-lived artifact of the early days of LLM and was not designed to be used in the production of a watermark.

The LLM watermark is very simple and useful. It is a simple and well-designed watermark that does not contain any watermarks. It is a watermark that will not hold anything, and it will not change when used in an LLM watermark. It is not a problem for everyone, but it is a problem for the rest of us.

In the absence of a watermark, you can simply mark the watermark with a pencil. You can use the pencil to mark a line, such as a line of code, or a line of code. You only need to mark code or code fragments using the pencil. Then, you can mark anything you want using the pencil, and that code can be used to
Text after normalization:

Explain in simple terms what an LLM watermark is and why it matters. It is a short-lived artifact of the early days of LLM and was not designed to be used in

In [11]:
attack_prompt = "Rewrite the following text using different wording but keep the same meaning:\n\n" + watermarked

inputs = tokenizer(attack_prompt, return_tensors="pt").to(device)
out = model.generate(
    **inputs,
    max_new_tokens=220,
    do_sample=True,
    temperature=0.9
)
paraphrased = tokenizer.decode(out[0], skip_special_tokens=True)

print("=== PARAPHRASED (ATTACK) ===")
print(paraphrased[-800:])
print("\nDetection (paraphrased):", detector.detect(paraphrased))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== PARAPHRASED (ATTACK) ===
he LMS watermarks. This can be seen on the left side of the image.

Watermark LMS watermarks are created with the same name and are found in all the LMS watermarks. To make a watermark in another image, you can use the left button to click the "Watermark" button.

Watermark on the right is the LMS watermark that appears on all the LMS watermarks. It says that it is the water mark that is the best candidate for a watermark. The watermark is found on the left side of the image.

Watermark on the left is the LMS watermark that is the lightest candidate for a watermark. This means that it has been created with the same name as the last of the LMS watermarks. It can be found on the left side of the image.


Watermark on the right is the LMS watermark that appears on all the LMS watermarks. This
Text after normalization:

Rewrite the following text using different wording but keep the same meaning:Explain in simple terms what an LLM watermark is and why it matter

In [4]:
!ls -la

total 16
drwxr-xr-x 1 root root 4096 Feb  6 14:31 .
drwxr-xr-x 1 root root 4096 Mar  3 22:20 ..
drwxr-xr-x 4 root root 4096 Feb  6 14:31 .config
drwxr-xr-x 1 root root 4096 Feb  6 14:31 sample_data
